# **Project Name**    - Facial Emotion Recognition Using Deep Learning



##### **Project Type** - Deep Learning / Computer Vision / Image Classification
##### **Contribution** - Individual
##### **Name** - Batti Chandan Singh

# **Project Summary -**

Write the summary here within 500-600 words.

Human emotions are a fundamental part of communication, and facial expressions often reveal a person’s emotional state without requiring spoken words. **DeepFER is an end-to-end deep-learning system designed to recognise human emotions from grayscale facial images.** The system classifies facial expressions into seven categories: **angry, disgust, fear, happy, neutral, sad, and surprise**. The project combines computer vision, image preprocessing, deep learning, evaluation, optimisation, and real-time prediction in a single reproducible workflow.

The project begins by understanding the structure and quality of the facial-image dataset. The analysis includes the total number of images, emotion classes, class distribution, supported file formats, image dimensions, and the number of images available for each emotion. Data-quality checks are performed to identify corrupted files, duplicate images, blurred samples, very dark or overexposed images, and images that may not contain a clearly visible face. **This early validation is important because poor-quality or imbalanced data can directly reduce recognition accuracy and create biased predictions toward dominant emotion classes.**

Since the dataset contains black-and-white images, the preprocessing pipeline uses **single-channel grayscale input** rather than unnecessary RGB information. Images are resized to a consistent shape, pixel values are normalised, and emotion labels are converted into numerical form. The dataset is divided into training, validation, and testing sets while preventing data leakage. Data augmentation techniques such as rotation, zoom, shifting, brightness variation, and horizontal flipping are applied only to the training set. **Applying augmentation only to training data improves generalisation while keeping validation and test results trustworthy.**

A Convolutional Neural Network is developed as the primary model because CNNs can automatically learn important facial features such as edges, eye shapes, eyebrow movement, mouth position, and expression-related textures. The project may also compare the CNN with a transfer-learning model. If a pretrained model requires three-channel input, the grayscale image can be repeated across three identical channels without introducing artificial colour information. Training callbacks such as early stopping, model checkpointing, and learning-rate reduction can be used to control overfitting and preserve the best-performing model.

Model performance is evaluated using **accuracy, precision, recall, F1-score, classification report, confusion matrix, and training-versus-validation accuracy and loss curves**. Hyperparameter optimisation is performed by adjusting the learning rate, batch size, dropout rate, optimiser, number of filters, dense-layer size, and training epochs. **The final model is selected not only by accuracy, but also by class-wise performance, validation stability, inference speed, and resistance to overfitting.**

A detailed error analysis is carried out using correctly classified, misclassified, and low-confidence predictions. This helps identify confusion between visually similar emotions and failures caused by blur, pose, lighting, or weak facial expressions. The best model is then saved, reloaded, and tested on unseen images to confirm reproducibility.

The final objective is to support **real-time facial emotion recognition using camera or video input**, where detected faces are processed consistently and assigned an emotion label with a confidence score. Such a system can contribute to human-computer interaction, customer-experience analysis, intelligent learning environments, virtual assistants, and emotion-aware applications. **DeepFER is not limited to training a classifier; it delivers a structured, explainable, reproducible, and deployment-ready facial emotion recognition pipeline.**

# **GitHub Link -**

https://github.com/BATTI-CHANDAN-SINGH/facial-recognition.git

# **Problem Statement**


**Write Problem Statement Here.**

Facial expressions provide valuable information about a person's emotional state. However, manually analysing emotions from a large collection of facial images or continuous video streams is **time-consuming, subjective, and difficult to scale**. Automatic emotion recognition is also challenging because expressions may vary due to lighting, pose, image quality, expression intensity, facial appearance, and similarities between emotion classes.

The objective of this project is to develop an **end-to-end deep-learning-based Facial Emotion Recognition system** that analyses a grayscale facial image and classifies it into one of seven emotions: **angry, disgust, fear, happy, neutral, sad, or surprise**.

The project applies **image-quality assessment, preprocessing, augmentation, Convolutional Neural Networks, hyperparameter optimization, multiclass evaluation, and error analysis** to create an accurate and reliable prediction model.

The final system should:

1. **Predict emotions from previously unseen facial images.**
2. **Support real-time emotion recognition from camera or video input.**
3. **Handle low-confidence predictions appropriately.**
4. **Save and reload the trained model without changing its performance.**
5. **Remain reproducible and executable from beginning to end without errors.**

The proposed system can support applications such as **human-computer interaction, customer-experience analysis, mental-health support, intelligent learning environments, virtual assistants, and personalised user experiences**.

# ***Let's Begin !***

## ***1. Know Your Image Dataset***

### Import Libraries

In [ ]:
import os
import random
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from PIL import Image, UnidentifiedImageError

# Suppress non-critical warnings
warnings.filterwarnings("ignore")

# Environment & Display Configurations
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Global Reproducibility
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# System Summary
print("Environment successfully initialized:")
print(f"  • TensorFlow : {tf.__version__}")
print(f"  • OpenCV     : {cv2.__version__}")
print(f"  • Seed       : {RANDOM_STATE}")

### Upload Dataset ZIP from Your Computer

In [ ]:
# Set the path of the ZIP file uploaded through the Colab Files panel

ZIP_PATH = Path("/content/DataSet.zip")

# Check whether the dataset file exists
if not ZIP_PATH.exists():
    raise FileNotFoundError(
        "DataSet.zip was not found. Upload it to the Colab Files panel."
    )

# Calculate the ZIP file size in MB
file_size = ZIP_PATH.stat().st_size / (1024 ** 2)

print("Dataset ZIP detected successfully.")
print("File name :", ZIP_PATH.name)
print("File path :", ZIP_PATH)
print(f"File size : {file_size:.2f} MB")

### Validate and Extract the Dataset

In [ ]:
# Create a folder for the extracted dataset

EXTRACT_PATH = Path("/content/deepfer_dataset")

# Remove the old folder when this cell is run again
if EXTRACT_PATH.exists():
    shutil.rmtree(EXTRACT_PATH)

EXTRACT_PATH.mkdir()

# Extract all files from the ZIP
with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:
    zip_file.extractall(EXTRACT_PATH)

print("Dataset extracted successfully.")
print("Extraction path:", EXTRACT_PATH)

# Display the main files or folders after extraction
for item in EXTRACT_PATH.iterdir():
    item_type = "Folder" if item.is_dir() else "File"
    print(f"[{item_type}] {item.name}")

### Detect Dataset Structure and Emotion Folders

In [ ]:
# Define the main training and testing folders

DATASET_ROOT = EXTRACT_PATH / "archive-3"
TRAIN_DIR = DATASET_ROOT / "train"
TEST_DIR = DATASET_ROOT / "test"

# Emotion classes used in the dataset
EMOTION_CLASSES = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
    "surprise"
]

# Check whether train and test folders are available
if not TRAIN_DIR.exists() or not TEST_DIR.exists():
    raise FileNotFoundError(
        "Train or test folder was not found in the extracted dataset."
    )

# Read the emotion folders available in each split
train_classes = sorted(
    folder.name
    for folder in TRAIN_DIR.iterdir()
    if folder.is_dir()
)

test_classes = sorted(
    folder.name
    for folder in TEST_DIR.iterdir()
    if folder.is_dir()
)

# Confirm that all seven emotion folders are available
if train_classes != sorted(EMOTION_CLASSES):
    raise ValueError(f"Unexpected training classes: {train_classes}")

if test_classes != sorted(EMOTION_CLASSES):
    raise ValueError(f"Unexpected testing classes: {test_classes}")

print("Dataset structure verified successfully.")
print("Training folder:", TRAIN_DIR)
print("Testing folder :", TEST_DIR)
print("Emotion classes:", train_classes)

### Assign and Validate Training and Testing Directories

In [ ]:
# Image formats supported in this project

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp"
}


# Count valid image files inside a folder
def count_images(folder):
    return sum(
        1
        for file in folder.iterdir()
        if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS
    )


# Count training and testing images for each emotion
train_counts = {
    emotion: count_images(TRAIN_DIR / emotion)
    for emotion in EMOTION_CLASSES
}

test_counts = {
    emotion: count_images(TEST_DIR / emotion)
    for emotion in EMOTION_CLASSES
}

# Create a table containing the image counts
class_count_df = pd.DataFrame({
    "Emotion": [emotion.capitalize() for emotion in EMOTION_CLASSES],
    "Training Images": train_counts.values(),
    "Testing Images": test_counts.values()
})

class_count_df["Total Images"] = (
    class_count_df["Training Images"]
    + class_count_df["Testing Images"]
)

display(class_count_df)

# Calculate the complete dataset size
total_train = sum(train_counts.values())
total_test = sum(test_counts.values())

print("Dataset summary:")
print("Number of classes      :", len(EMOTION_CLASSES))
print(f"Total training images : {total_train:,}")
print(f"Total testing images  : {total_test:,}")
print(f"Complete dataset size : {total_train + total_test:,}")

### Dataset First View

The following visualization displays one sample grayscale image from each emotion class to verify that the images and labels are loaded correctly.

In [ ]:
# Display one sample image from each emotion class

fig, axes = plt.subplots(1, 7, figsize=(15, 3))

sample_details = []

for axis, emotion in zip(axes, EMOTION_CLASSES):

    # Select the first image from the emotion folder
    image_path = next(
        file
        for file in (TRAIN_DIR / emotion).iterdir()
        if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS
    )

    # Read the image in grayscale format
    image = cv2.imread(
        str(image_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        raise ValueError(f"Unable to read image: {image_path}")

    # Display the selected image
    axis.imshow(image, cmap="gray")
    axis.set_title(emotion.capitalize())
    axis.axis("off")

    # Save basic details about the image
    sample_details.append({
        "Emotion": emotion.capitalize(),
        "File Name": image_path.name,
        "Height": image.shape[0],
        "Width": image.shape[1],
        "Channels": 1
    })

plt.suptitle(
    "Sample Images from Each Emotion Class",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

# Display the details of the sample images
sample_details_df = pd.DataFrame(sample_details)
display(sample_details_df)

### What did you know about your dataset?


- The dataset contains **35,887 facial images**.
- It is divided into **28,709 training images** and **7,178 testing images**.
- The dataset contains seven emotion classes: **Angry, Disgust, Fear, Happy, Neutral, Sad, and Surprise**.
- All images are grayscale with a size of **48 × 48 pixels** and one channel.
- The dataset is imbalanced because some emotions contain more images than others.
- The Happy class has the highest number of training images, while the Disgust class has the lowest.

## ***2.  Understanding Your Images and Labels***

### 2.1 Emotion Class Distribution

The following graph compares the number of training and testing images available in each emotion class. It helps identify whether the dataset is balanced or imbalanced.

In [ ]:
# Compare training and testing images for each emotion class

x = np.arange(len(EMOTION_CLASSES))
bar_width = 0.35

plt.figure(figsize=(12, 5))

# Plot training and testing image counts
plt.bar(
    x - bar_width / 2,
    class_count_df["Training Images"],
    width=bar_width,
    label="Training"
)

plt.bar(
    x + bar_width / 2,
    class_count_df["Testing Images"],
    width=bar_width,
    label="Testing"
)

# Add labels and title
plt.xticks(x, class_count_df["Emotion"])
plt.xlabel("Emotion Classes")
plt.ylabel("Number of Images")
plt.title("Emotion Class Distribution")
plt.legend()

plt.tight_layout()
plt.show()

### 2.2 Image Properties

This step checks the image dimensions, colour channels, file formats, and unreadable images in the complete dataset.

In [ ]:
# Check the basic properties of all images

image_sizes = set()
image_channels = set()
image_formats = set()
unreadable_images = 0

# Check both training and testing folders
for dataset_folder in [TRAIN_DIR, TEST_DIR]:

    for emotion in EMOTION_CLASSES:

        emotion_folder = dataset_folder / emotion

        for image_path in emotion_folder.iterdir():

            # Skip files that are not supported images
            if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue

            # Read the image without changing its original format
            image = cv2.imread(str(image_path), cv2.IMREAD_UNCHANGED)

            if image is None:
                unreadable_images += 1
                continue

            # Store image height and width
            image_sizes.add(image.shape[:2])

            # Identify grayscale or colour channels
            channels = 1 if len(image.shape) == 2 else image.shape[2]
            image_channels.add(channels)

            # Store the image file format
            image_formats.add(image_path.suffix.lower())

# Display the collected image properties
print("Image sizes       :", image_sizes)
print("Image channels    :", image_channels)
print("Image formats     :", image_formats)
print("Unreadable images :", unreadable_images)

### 2.3 Create Image Metadata

A metadata table is created for all training and testing images. It contains the image name, emotion label, dataset split, dimensions, format, and file size.

In [ ]:
# Create metadata for all training and testing images

metadata = []

for split, folder in [("Train", TRAIN_DIR), ("Test", TEST_DIR)]:

    for emotion in EMOTION_CLASSES:

        for image_path in (folder / emotion).iterdir():

            # Include only supported image files
            if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue

            # Store the important details of each image
            metadata.append({
                "File Name": image_path.name,
                "Emotion": emotion.capitalize(),
                "Split": split,
                "Height": 48,
                "Width": 48,
                "Channels": 1,
                "Format": image_path.suffix.lower(),
                "File Size (KB)": round(
                    image_path.stat().st_size / 1024, 2
                ),
                "Image Path": str(image_path)
            })

# Convert the collected details into a DataFrame
image_metadata_df = pd.DataFrame(metadata)

print("Metadata created successfully.")
print("Total metadata records:", len(image_metadata_df))

# Display the first five records
display(image_metadata_df.head())

### 2.4 Check Emotion Labels and Dataset Splits

This step verifies the unique emotion labels and the number of images available in the training and testing datasets.

In [ ]:
# Check unique emotion labels and dataset split counts

emotion_labels = image_metadata_df["Emotion"].unique()
split_counts = image_metadata_df["Split"].value_counts()

print("Number of emotion classes:", len(emotion_labels))
print("Emotion labels:", sorted(emotion_labels))

print("\nImages in each split:")
print(split_counts)

### 2.5 Class Imbalance Analysis

This step compares the number of training images in each emotion class and calculates the imbalance ratio between the largest and smallest classes.

In [ ]:
# Analyze class imbalance in the training dataset

imbalance_df = class_count_df[
    ["Emotion", "Training Images"]
].copy()

# Calculate each class percentage
imbalance_df["Percentage"] = (
    imbalance_df["Training Images"] / total_train * 100
).round(2)

# Sort classes from highest to lowest
imbalance_df = imbalance_df.sort_values(
    "Training Images",
    ascending=False
).reset_index(drop=True)

# Calculate the imbalance ratio
highest_count = imbalance_df["Training Images"].max()
lowest_count = imbalance_df["Training Images"].min()
imbalance_ratio = highest_count / lowest_count

display(imbalance_df)

print(f"Imbalance ratio: {imbalance_ratio:.2f}:1")
print("Most represented class :", imbalance_df.iloc[0]["Emotion"])
print("Least represented class:", imbalance_df.iloc[-1]["Emotion"])

### 2.6 Variables Description

| Variable | Description |
|---|---|
| **File Name** | Name of the facial image file. |
| **Emotion** | Emotion label assigned to the image. |
| **Split** | Indicates whether the image belongs to the training or testing dataset. |
| **Height** | Height of the image in pixels. All images have a height of 48 pixels. |
| **Width** | Width of the image in pixels. All images have a width of 48 pixels. |
| **Channels** | Number of image channels. The value is 1 because the images are grayscale. |
| **Format** | File format of the image. All images are stored in JPG format. |
| **File Size (KB)** | Storage size of the image in kilobytes. |
| **Image Path** | Complete location of the image inside the dataset folder. |

### 2.7 What Did You Know About Your Images and Labels?

- The dataset contains **35,887 facial images** with seven emotion labels.
- Every image has the same size of **48 × 48 pixels**.
- All images are grayscale and contain only **one channel**.
- All image files are stored in **JPG format**.
- No unreadable or corrupted images were found.
- The training dataset contains **28,709 images**, while the testing dataset contains **7,178 images**.
- The dataset is highly imbalanced with an imbalance ratio of **16.55:1**.
- **Happy** is the most represented emotion with 7,215 training images.
- **Disgust** is the least represented emotion with only 436 training images.
- Class weights and data augmentation will be used later to reduce the effect of class imbalance.

## 3. ***Image Data Wrangling and Quality Checking***

### 3.1 Check Missing and Duplicate Images

This step checks whether any image files are missing from their recorded paths and identifies exact duplicate files in the dataset.

In [ ]:
# Check missing files and exact duplicate images

missing_files = []
duplicate_files = []
seen_images = {}

for image_path in image_metadata_df["Image Path"]:

    path = Path(image_path)

    # Record files that are missing from the dataset
    if not path.exists():
        missing_files.append(image_path)
        continue

    # Create a signature using the file content
    file_data = path.read_bytes()
    signature = (len(file_data), hash(file_data))

    # Check whether the same file content appeared earlier
    if signature in seen_images:
        duplicate_files.append({
            "Duplicate Image": image_path,
            "Original Image": seen_images[signature]
        })
    else:
        seen_images[signature] = image_path

# Convert duplicate details into a DataFrame
duplicate_df = pd.DataFrame(duplicate_files)

print("Missing image files :", len(missing_files))
print("Duplicate images    :", len(duplicate_df))

# Display a few duplicate records when available
if not duplicate_df.empty:
    display(duplicate_df.head())
else:
    print("No exact duplicate images were found.")

### 3.2 Analyse Duplicate Images

This step checks whether duplicate images occur within the same dataset split or between the training and testing datasets. It also checks whether duplicate images have the same emotion label.

In [ ]:
# Create mappings for split and emotion using the image path

split_map = image_metadata_df.set_index("Image Path")["Split"].to_dict()
emotion_map = image_metadata_df.set_index("Image Path")["Emotion"].to_dict()

# Add split details for original and duplicate images
duplicate_df["Original Split"] = duplicate_df["Original Image"].map(split_map)
duplicate_df["Duplicate Split"] = duplicate_df["Duplicate Image"].map(split_map)

# Add emotion details for original and duplicate images
duplicate_df["Original Emotion"] = duplicate_df["Original Image"].map(emotion_map)
duplicate_df["Duplicate Emotion"] = duplicate_df["Duplicate Image"].map(emotion_map)

# Identify duplicates within the same split or across train and test
duplicate_df["Duplicate Type"] = np.where(
    duplicate_df["Original Split"] == duplicate_df["Duplicate Split"],
    "Within Same Split",
    "Train-Test Overlap"
)

# Check whether duplicate images have the same label
duplicate_df["Label Match"] = np.where(
    duplicate_df["Original Emotion"] == duplicate_df["Duplicate Emotion"],
    "Same Label",
    "Different Label"
)

# Create a summary of duplicate types
duplicate_summary = (
    duplicate_df
    .groupby(["Duplicate Type", "Label Match"])
    .size()
    .reset_index(name="Count")
)

display(duplicate_summary)

print(
    "\nTrain-test duplicate images:",
    (duplicate_df["Duplicate Type"] == "Train-Test Overlap").sum()
)

print(
    "Duplicates with different labels:",
    (duplicate_df["Label Match"] == "Different Label").sum()
)

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.

### What all manipulations have you done and insights you found?

Answer Here.

## ***4. Image Visualization, Storytelling and Analysis***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 2

In [ ]:
# Chart - 2 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 3

In [ ]:
# Chart - 3 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 4

In [ ]:
# Chart - 4 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 5

In [ ]:
# Chart - 5 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 6

In [ ]:
# Chart - 6 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 7

In [ ]:
# Chart - 7 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 8

In [ ]:
# Chart - 8 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 9

In [ ]:
# Chart - 9 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 10

In [ ]:
# Chart - 10 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 11

In [ ]:
# Chart - 11 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 12

In [ ]:
# Chart - 12 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 13

In [ ]:
# Chart - 13 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

## ***Face Detection and Image Preprocessing*** **bold text**

## **Feature Engineering and Dataset Preparation**

## **Facial Recognition Model Implementation**

## **Future Work and Deployment**

# **Conclusion**

Write the conclusion here.

### ***Hurrah! You have successfully completed your Facial Recognition Project !!!***